[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/hypothesis_testing.ipynb)

# Hypothesis testing full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the standard tests, several steps each), **tier 3** stretch problems (combine ideas, spot
the trap, work backwards) and **tier 4**, the case: all 400 households of Wrenmoor Energy's trial of an in-home
display (a fictional company; our own data), and the question of whether the display cuts electricity use enough to
pay for itself. Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same tests with `scipy.stats`, `numpy` and
`pandas`, so you can run them on your own data. And a few cells you can move yourself: a sample mean that slides
while you watch its p-value, a sample size that bends the power curve, and the A/B test whose lift stays the same
while the visitors grow.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Decimals as
decimals: `0.0228`, not `2.28%`; a p-value to four decimals is close enough. Numbers without thousands separators:
`8155`, not `8,155`. You can type the arithmetic too: `(103-100)/(12/sqrt(36))`. For several answers, separate them
with commas, in the order the question lists them (the box's label shows the order). Letters and words as they are:
`B`, `Type I`. You can also put your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run
the cell: it checks straight away.

**The cheat sheet.** Every test, formula and spreadsheet function from the course on two pages:
[hypothesis testing cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/hypothesis_testing_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, math, operator, re, random, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# the H0 curve CHALK · the true-mean curve SKY · rejection region VIOLET · the p-value GOLD; display MINT, control SKY
PAIR = {"display": MINT, "control": SKY}

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(kwh|kpa|mm|minutes?|mins?|months?|hours?|days?|visitors?|cycles?)\b|[$%]"
SAME = {"b": {"b", "option b", "pair b"}, "type i": {"type i", "type 1", "type one", "i", "1"},
        "47, 54": {"47, 54", "47 and 54"}}

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    if isinstance(want, str):
        g = got if isinstance(got, str) else f"{got:g}"
        g = re.sub(r"\s*,\s*", ", ", g)
        return g == want.lower() or g in SAME.get(want.lower(), ())
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Hypotheses and errors',
     "q": 'Order values have σ = $18. What is the standard error of the mean of a sample of n = 36 orders?',
     "parts": ['standard error ($)'],
     "answer": [3.0], "tol": [0.005], "function": 'SQRT',
     "working": 'σ/√n = 18/6 = 3'},
    {"id": 'P1.2', "tier": 1, "group": 'Hypotheses and errors',
     "q": 'A call centre claims its mean wait is at most 2 minutes. You want to challenge the claim. Which pair is right? A: H0 μ ≥ 2, Ha μ < 2. B: H0 μ ≤ 2, Ha μ > 2. C: H0 μ = 2, Ha μ ≠ 2.',
     "parts": ['type one letter: A, B or C'],
     "answer": ['B'], "tol": [0], "function": '—',
     "working": 'The claim is the null: μ ≤ 2. The challenge is the alternative: μ > 2.'},
    {"id": 'P1.3', "tier": 1, "group": 'Hypotheses and errors',
     "q": "H0: the new packing robot is no faster. The test concludes it is faster, but in truth it isn't. Type I or Type II error?",
     "parts": ['type: Type I or Type II'],
     "answer": ['Type I'], "tol": [0], "function": '—',
     "working": 'Rejecting a true null hypothesis is a Type I error.'},
    {"id": 'P1.4', "tier": 1, "group": 'Testing a mean: sigma known',
     "q": 'x̄ = 103, μ0 = 100, σ = 12, n = 36. Compute the test statistic z.',
     "parts": ['z'],
     "answer": [1.5], "tol": [0.005], "function": 'SQRT',
     "working": 'z = (103 − 100)/(12/√36) = 3/2 = 1.5'},
    {"id": 'P1.5', "tier": 1, "group": 'Testing a mean: sigma known',
     "q": 'A lower-tail test gives z = −1.9. What is the p-value?',
     "parts": ['p-value'],
     "answer": [0.028716559816001817], "tol": [5e-05], "function": 'NORM.S.DIST',
     "working": 'p = P(Z ≤ −1.9) = NORM.S.DIST(−1.9, TRUE) ≈ 0.0287'},
    {"id": 'P1.6', "tier": 1, "group": 'Testing a mean: sigma known',
     "q": 'What are the critical values of a two-tailed z test at α = 0.01?',
     "parts": ['critical value (the positive one)'],
     "answer": [2.5758293035488986], "tol": [0.0005], "function": 'NORM.S.INV',
     "working": '±z for 0.005 in each tail: NORM.S.INV(0.995) ≈ 2.576'},
    {"id": 'P1.7', "tier": 1, "group": 'Sigma unknown and proportions',
     "q": 'x̄ = 52, μ0 = 50, s = 6, n = 16. Compute t and its degrees of freedom.',
     "parts": ['t', 'degrees of freedom'],
     "answer": [1.3333333333333333, 15], "tol": [0.0005, 0], "function": 'SQRT',
     "working": 't = 2/(6/√16) = 2/1.5 ≈ 1.333; df = 15'},
    {"id": 'P1.8', "tier": 1, "group": 'Sigma unknown and proportions',
     "q": 'A sample of n = 400 gives p̄ = 0.46. Test H0: p = 0.5. Compute z.',
     "parts": ['z'],
     "answer": [-1.5999999999999992], "tol": [0.0005], "function": 'SQRT',
     "working": 'σ_p̄ = √(0.5 × 0.5/400) = 0.025; z = −0.04/0.025 = −1.6'},
    {"id": 'P1.9', "tier": 1, "group": 'Two population means',
     "q": 'σ1 = 10, n1 = 50, σ2 = 12, n2 = 40. What is the standard error of x̄1 − x̄2?',
     "parts": ['standard error of x̄₁ − x̄₂'],
     "answer": [2.3664319132398464], "tol": [0.0005], "function": 'SQRT',
     "working": '√(100/50 + 144/40) = √5.6 ≈ 2.366'},
    {"id": 'P1.10', "tier": 1, "group": 'Matched samples and two proportions',
     "q": 'Version A: 120 sign-ups from 1,000 visitors. Version B: 150 from 1,000. What is the pooled proportion p̄ for the test of p1 − p2 = 0?',
     "parts": ['pooled proportion p̄ (a decimal)'],
     "answer": [0.135], "tol": [5e-05], "function": 'SUM',
     "working": 'p̄ = (120 + 150)/(1000 + 1000) = 0.135'},
    {"id": 'P2.1', "tier": 2, "group": 'Testing a mean: sigma known',
     "q": 'A warehouse claims its mean pick time is at most 5.0 minutes. σ = 1.2 min (long history). A sample of 36 picks averages 5.4 min. Test the claim at α = 0.05: give z and the p-value, and decide.',
     "parts": ['z', 'p-value'],
     "answer": [2.0000000000000018, 0.022750131948179098], "tol": [0.005, 0.0005], "function": 'NORM.S.DIST',
     "working": 'H0 μ ≤ 5.0, Ha μ > 5.0; SE = 1.2/6 = 0.2; z = 0.4/0.2 = 2.0; p = 1 − NORM.S.DIST(2, TRUE) ≈ 0.0228 < 0.05: reject; the mean pick time exceeds 5 minutes'},
    {"id": 'P2.2', "tier": 2, "group": 'Testing a mean: sigma known',
     "q": 'A machine bores holes to a target of 20.00 mm; σ = 0.05 mm. 25 holes average 20.02 mm. Two-tailed test at α = 0.05: the p-value, and the 95% confidence interval for μ.',
     "parts": ['p-value (two-tailed)', '95% interval: lower end (mm)', '95% interval: upper end (mm)'],
     "answer": [0.04550026389636308, 20.0004003601546, 20.0395996398454], "tol": [0.0005, 0.0005, 0.0005], "function": 'NORM.S.DIST, NORM.S.INV',
     "working": 'SE = 0.05/5 = 0.01; z = 2.0; p = 2 × (1 − NORM.S.DIST(2, TRUE)) ≈ 0.0455: reject; interval 20.02 ± 1.96 × 0.01 = 20.0004 to 20.0396, which excludes 20.00'},
    {"id": 'P2.3', "tier": 2, "group": 'Sigma unknown and proportions',
     "q": 'Is the mean first-month spend of new subscribers above $40? n = 20, x̄ = $43.10, s = $6.20. α = 0.05: give t and the p-value.',
     "parts": ['t', 'p-value'],
     "answer": [2.2360679774997907, 0.018770274774262408], "tol": [0.0005, 0.0005], "function": 'T.DIST.RT',
     "working": 'Ha μ > 40; SE = 6.2/√20 ≈ 1.386; t = 3.1/1.386 ≈ 2.236, df 19; p = T.DIST.RT(2.236, 19) ≈ 0.019: reject'},
    {"id": 'P2.4', "tier": 2, "group": 'Sigma unknown and proportions',
     "q": 'A pressure sensor should read 100.0 kPa against a reference. Eight readings: 100.4, 99.8, 100.9, 100.6, 100.2, 101.1, 99.9, 100.7. Is it biased? Two-tailed, α = 0.05: give t and the p-value.',
     "parts": ['t', 'p-value (two-tailed)'],
     "answer": [2.749545416973528, 0.028522667328768403], "tol": [0.0005, 0.0005], "function": 'AVERAGE, STDEV.S, T.DIST.2T',
     "working": 'x̄ = 100.45, s ≈ 0.463; t = 0.45/(0.463/√8) ≈ 2.750, df 7; p = T.DIST.2T(2.750, 7) ≈ 0.029: reject; it reads high'},
    {"id": 'P2.5', "tier": 2, "group": 'Sigma unknown and proportions',
     "q": 'A courier claims at least 90% of parcels arrive on time. In a sample of 520, 450 did. Test the claim at α = 0.05: z and the p-value.',
     "parts": ['z', 'p-value'],
     "answer": [-2.6311740579210867, 0.004254522202430666], "tol": [0.0005, 0.0005], "function": 'NORM.S.DIST',
     "working": 'H0 p ≥ 0.9; p̄ = 450/520 ≈ 0.8654; σ_p̄ = √(0.09/520) ≈ 0.01316; z ≈ −2.63; p = NORM.S.DIST(z, TRUE) ≈ 0.0043: reject the claim'},
    {"id": 'P2.6', "tier": 2, "group": 'Decisions, power and sample size',
     "q": 'H0: μ ≤ 120 vs Ha: μ > 120, σ = 15, n = 25, α = 0.05. If the true mean is 127, what are β (the probability of a Type II error) and the power?',
     "parts": ['β', 'power'],
     "answer": [0.24557537274577299, 0.754424627254227], "tol": [0.0005, 0.0005], "function": 'NORM.S.INV, NORM.S.DIST',
     "working": 'Reject if x̄ > 120 + 1.645 × 3 = 124.93; β = P(x̄ ≤ 124.93 | μ = 127) = NORM.S.DIST((124.93 − 127)/3, TRUE) ≈ 0.246; power ≈ 0.754'},
    {"id": 'P2.7', "tier": 2, "group": 'Decisions, power and sample size',
     "q": 'How many observations does a one-tailed test need, with α = 0.05, β = 0.10, σ = 8, to detect a mean 3 units from μ0?',
     "parts": ['n (rounded up)'],
     "answer": [61], "tol": [0], "function": 'NORM.S.INV, ROUNDUP',
     "working": 'n = (1.645 + 1.282)² × 64/9 ≈ 60.9 → round up to 61'},
    {"id": 'P2.8', "tier": 2, "group": 'Two population means',
     "q": 'Call handling time (minutes) after two training methods: A n = 15, x̄ = 6.8, s = 1.2; B n = 18, x̄ = 6.1, s = 0.9. Two-tailed Welch test: t, degrees of freedom, p.',
     "parts": ['t', 'degrees of freedom (one decimal)', 'p-value (two-tailed)'],
     "answer": [1.8641827445195764, 25.573596653370952, 0.07381483264519306], "tol": [0.0005, 0.05, 0.0005], "function": 'SQRT, T.DIST.2T',
     "working": 'SE = √(1.44/15 + 0.81/18) ≈ 0.3755; t = 0.7/0.3755 ≈ 1.864; df ≈ 25.6; p ≈ 0.074: not significant at 0.05'},
    {"id": 'P2.9', "tier": 2, "group": 'Matched samples and two proportions',
     "q": "Ten sales reps' weekly sales ($ thousands) before and after a negotiation course. Before: 42, 38, 51, 45, 36, 48, 40, 55, 44, 39. After: 45, 41, 50, 49, 40, 52, 41, 58, 47, 41. Did sales rise? Matched-samples test, α = 0.05: d̄, t, p.",
     "parts": ['d̄ ($ thousands)', 't', 'p-value'],
     "answer": [2.6, 5.211594217292276, 0.000277695333230521], "tol": [0.005, 0.0005, 0.0005], "function": 'AVERAGE, STDEV.S, T.DIST.RT, T.TEST',
     "working": 'd = after − before: 3, 3, −1, 4, 4, 4, 1, 3, 3, 2; d̄ = 2.6, s_d ≈ 1.578; t = 2.6/(1.578/√10) ≈ 5.21, df 9; p ≈ 0.0003: reject'},
    {"id": 'P2.10', "tier": 2, "group": 'Matched samples and two proportions',
     "q": 'Two email subject lines, 2,500 recipients each: A gets 125 clicks, B gets 160. Two-tailed test of p1 − p2 = 0 at α = 0.05: pooled p̄, z, p.',
     "parts": ['pooled p̄', "z (B's rate minus A's)", 'p-value (two-tailed)'],
     "answer": [0.057, 2.1349601246546537, 0.03276426741606686], "tol": [5e-05, 0.0005, 0.0005], "function": 'NORM.S.DIST, SQRT',
     "working": 'p̄ = 285/5000 = 0.057; SE = √(0.057 × 0.943 × 2/2500) ≈ 0.00656; z = 0.014/0.00656 ≈ 2.135; p ≈ 0.033: reject'},
    {"id": 'P3.1', "tier": 3, "group": 'Testing a mean: sigma known',
     "q": "A test planned as two-tailed gives z = 1.81, p ≈ 0.070. After seeing that x̄ is above the target, a colleague re-runs it one-tailed and reports p ≈ 0.035, 'significant'. What p-value should the report carry, and why?",
     "parts": ['the p-value to report'],
     "answer": [0.0702957871680776], "tol": [0.0005], "function": 'NORM.S.DIST',
     "working": 'The tails are chosen before seeing the data. Switching after doubles the real chance of a false alarm. Report the planned two-tailed p ≈ 0.070: not significant at 0.05.'},
    {"id": 'P3.2', "tier": 3, "group": 'Testing a mean: sigma known',
     "q": 'A 95% confidence interval for a mean (σ known) runs from 48.2 to 53.8. Work backwards: what are x̄ and the standard error? At α = 0.05 (two-tailed), which of μ0 = 47, 50, 54 are rejected?',
     "parts": ['x̄', 'standard error', 'the rejected values, smallest first (a comma and a space between)'],
     "answer": [51.0, 1.4285976793890307, '47, 54'], "tol": [0.005, 0.0005, 0], "function": 'NORM.S.INV',
     "working": 'x̄ = midpoint 51.0; half-width 2.8 = 1.96 × SE, so SE ≈ 1.429; reject exactly the values outside the interval: 47 and 54 (z = 2.80 and −2.10)'},
    {"id": 'P3.3', "tier": 3, "group": 'Sigma unknown and proportions',
     "q": 'A one-tailed t test gives t = 2.1 with 15 degrees of freedom. What is the p-value, and what is the smallest α at which H0 would be rejected?',
     "parts": ['p-value (= the smallest α that rejects)'],
     "answer": [0.02652762807602138], "tol": [0.0005], "function": 'T.DIST.RT',
     "working": 'p = T.DIST.RT(2.1, 15) ≈ 0.0265; that is also the smallest α that rejects'},
    {"id": 'P3.4', "tier": 3, "group": 'Decisions, power and sample size',
     "q": 'A battery cell test: H0 μ ≥ 500 cycles, σ = 40, α = 0.05. With n = 64 the power at μ = 490 is about 0.64. What is it with n = 144?',
     "parts": ['power with n = 144'],
     "answer": [0.9123145367502967], "tol": [0.0005], "function": 'NORM.S.INV, NORM.S.DIST',
     "working": 'SE = 40/12 ≈ 3.333; reject if x̄ < 500 − 1.645 × 3.333 ≈ 494.52; power = P(x̄ < 494.52 | μ = 490) ≈ 0.912'},
    {"id": 'P3.5', "tier": 3, "group": 'Decisions, power and sample size',
     "q": 'An online shop tests two layouts with 400,000 visitors each. Mean basket: $64.20 vs $64.05, s = $25 in both. Is the difference statistically significant (two-tailed)? Give the 95% interval. A change is worth making only if it lifts the basket by at least $0.50: is it practically significant?',
     "parts": ['p-value (two-tailed)', '95% interval: lower end ($)', '95% interval: upper end ($)'],
     "answer": [0.007290358091535776, 0.04043468242792726, 0.25956531757207274], "tol": [0.0005, 0.005, 0.005], "function": 'NORM.S.DIST, NORM.S.INV',
     "working": 'SE = 25 × √(2/400000) ≈ 0.0559; z ≈ 2.68, p ≈ 0.007: significant; interval $0.04 to $0.26, entirely below $0.50: not practically significant'},
    {"id": 'P3.6', "tier": 3, "group": 'Two population means',
     "q": "Two suppliers' delivery delays (hours): A (10 orders) 82, 104, 95, 58, 116, 74, 93, 101, 67, 80; B (20 orders) 78, 74, 81, 69, 76, 72, 80, 75, 77, 73, 79, 70, 82, 74, 76, 71, 78, 75, 73, 77. Compare the pooled-variance t test with Welch's (two-tailed). Which should you report, and why?",
     "parts": ['pooled test: p-value', 'Welch: p-value', 'Welch: degrees of freedom (one decimal)'],
     "answer": [0.009215949146259783, 0.07501838280860706, 9.360741165988829], "tol": [0.0005, 0.0005, 0.05], "function": 'T.TEST (type 2 and type 3)',
     "working": "Pooled: t ≈ 2.80, df 28, p ≈ 0.009. Welch: t ≈ 2.00, df ≈ 9.4, p ≈ 0.075. A's spread is far larger and its sample smaller, so the pooled test understates the uncertainty: report Welch's"},
    {"id": 'P3.7', "tier": 3, "group": 'Matched samples and two proportions',
     "q": "Use P2.9's sales data, but analyse it as two independent samples (Welch). What p-value do you get (two-tailed), and why is it so different from the matched-samples test?",
     "parts": ['Welch p-value (two-tailed)'],
     "answer": [0.3454113398599665], "tol": [0.0005], "function": 'T.TEST (type 1 vs type 3)',
     "working": 'Welch: t ≈ 0.97, p ≈ 0.35. The reps differ far more from each other (s ≈ 6) than their change varies (s_d ≈ 1.6). Pairing removes the rep-to-rep spread, so the matched test sees the rise clearly'},
    {"id": 'P3.8', "tier": 3, "group": 'Matched samples and two proportions',
     "q": 'A sign-up page converts 5%. How many visitors per version does an A/B test need to detect a rise to 6%, two-tailed α = 0.05, power 0.80?',
     "parts": ['visitors per version (rounded up)'],
     "answer": [8155], "tol": [0], "function": 'NORM.S.INV, ROUNDUP',
     "working": 'n = (1.960 + 0.842)² × (0.05 × 0.95 + 0.06 × 0.94)/0.01² ≈ 8,155.0 → 8,155 per version'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": 'Reproduce the case: the mean change in monthly use (after − before) for display and control households, and the extra saving of the display group.',
     "parts": ['display: mean change (kWh)', 'control: mean change (kWh)', 'extra saving of the display (kWh)'],
     "answer": [-78.78, -48.43, 30.35], "tol": [0.05, 0.05, 0.05], "function": 'AVERAGEIF',
     "working": 'AVERAGEIF on a change column: display ≈ −78.8, control ≈ −48.4; extra saving = 30.35 kWh a month'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": "The first tempting analysis: compare this winter's use only (after_kwh), display vs control, with Welch's test (two-tailed). What p-value do you get, and why does it miss the effect?",
     "parts": ['p-value (two-tailed)'],
     "answer": [0.4435703301982019], "tol": [0.005], "function": 'T.TEST (type 3)',
     "working": 'Means ≈ 793.6 vs 816.9; SE ≈ 30.4; t ≈ −0.77, p ≈ 0.44. Households differ by hundreds of kWh, so a 30 kWh effect drowns in that spread'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": "The right analysis: Welch's test on the changes, one-tailed as planned (Ha: the display group's change is lower). Give t, the degrees of freedom and the p-value.",
     "parts": ['t', 'degrees of freedom', 'p-value (one-tailed)'],
     "answer": [-4.906389990164745, 397.21319852375655, 6.779782041314022e-07], "tol": [0.005, 0.5, 1e-06], "function": 'T.TEST (type 3, one tail)',
     "working": 'SE = √(s1²/200 + s2²/200) ≈ 6.19; t ≈ −4.91, df ≈ 397; p ≈ 0.0000007: reject'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": 'Give the 95% confidence interval for the extra saving (kWh a month), and in dollars a month at $0.16 per kWh.',
     "parts": ['95% interval: lower end (kWh a month)', '95% interval: upper end (kWh a month)'],
     "answer": [18.188979344747743, 42.51102065525226], "tol": [0.05, 0.05], "function": 'T.INV.2T',
     "working": '30.35 ± 1.966 × 6.19: about 18.2 to 42.5 kWh, or $2.91 to $6.80 a month'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": 'Practical significance: before the trial the manager set 15 kWh a month as the saving that pays the display back in two years. Test H0: saving ≤ 15 against Ha: saving > 15. t and p?',
     "parts": ['t', 'p-value'],
     "answer": [2.4814855469202253, 0.006748198823668394], "tol": [0.005, 0.0005], "function": 'T.DIST.RT',
     "working": 't = (30.35 − 15)/6.19 ≈ 2.48; p = T.DIST.RT(2.48, 397) ≈ 0.007: the saving clears the bar'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": "A second question: is the saving the same in houses and apartments? Estimate the extra saving and its 95% interval in each segment, and the saving as a percent of the segment's usual use.",
     "parts": ['houses: extra saving (kWh a month)', 'apartments: extra saving (kWh a month)'],
     "answer": [35.68811188811189, 25.129049531459167], "tol": [0.05, 0.05], "function": 'AVERAGEIFS, T.INV.2T',
     "working": 'Houses: ≈ 35.7 kWh (19.1 to 52.3), 3.4%; apartments: ≈ 25.1 kWh (7.8 to 42.5), 3.9%. In kWh, houses save more; as a share of use the two are close. The segment was not planned, so treat it as a lead to confirm, not a finding'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": 'Another view: the share of households that cut their use by at least 5%. Compare display and control with a two-proportion test (one-tailed): the two shares, z and p.',
     "parts": ['display: share cutting ≥ 5% (a decimal)', 'control: share cutting ≥ 5% (a decimal)', 'z', 'p-value (one-tailed)'],
     "answer": [0.74, 0.545, 4.06874078482604, 2.363394877447078e-05], "tol": [0.0005, 0.0005, 0.005, 0.0005], "function": 'COUNTIFS, NORM.S.DIST',
     "working": 'display 148/200, control 109/200; pooled p̄ = 0.6425; z ≈ 4.07; p ≈ 0.0000'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Wrenmoor Energy',
     "q": 'Write the recommendation. Anchor it: at the low end of the 95% interval, how many months does a display take to pay back ($45 each, plus $0.50 a month for the report)?',
     "parts": ["months to pay back, at the interval's low end"],
     "answer": [18.67036548334498], "tol": [0.05], "function": '—',
     "working": 'Low end ≈ 18.2 kWh × $0.16 ≈ $2.91 a month, minus $0.50 → $2.41; 45/2.41 ≈ 18.7 months. Recommendation: roll the display out; even the cautious estimate pays back within two years. Limits: one winter, one region, three months; keep a control group for a year to check the saving lasts'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def split(text, q):
    """The typed answer, one part per comma; a last answer that is itself a list ("47, 54") keeps its commas."""
    got = [read_part(p) for p in re.split(r"[,;]", text) if p.strip()]
    n, last = len(q["answer"]), q["answer"][-1]
    if len(got) > n and isinstance(last, str) and "," in last:
        got = got[:n - 1] + [", ".join(g if isinstance(g, str) else f"{g:g}" for g in got[n - 1:])]
    return got

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = split(box.value, q)
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

## Tier 1 · Warm-up

One idea, one step, small numbers: a standard error, writing H₀ and Hₐ, naming an error, a z statistic, a p-value, a
critical value, a t statistic with its degrees of freedom, a z for a proportion, the standard error of a difference
and a pooled proportion.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

`scipy.stats.norm` is the standard normal curve: `norm.cdf(z)` is the area to the left of z (a spreadsheet's
NORM.S.DIST(z, TRUE)), `norm.sf(z)` the area to the right, and `norm.ppf(p)` the z with area p to its left
(NORM.S.INV). A standard error is one line of arithmetic. Each line below computes one warm-up answer and checks it
against the practice set.

In [ ]:
from scipy.stats import norm

se = 18 / math.sqrt(36)                                          # P1.1
z = (103 - 100) / (12 / math.sqrt(36))                           # P1.4
p = norm.cdf(-1.9)                                               # P1.5: lower tail
crit = norm.ppf(1 - 0.01 / 2)                                    # P1.6: two-tailed, α = 0.01
t, df = (52 - 50) / (6 / math.sqrt(16)), 16 - 1                  # P1.7
z_prop = (0.46 - 0.5) / math.sqrt(0.5 * 0.5 / 400)               # P1.8
se_diff = math.sqrt(10 ** 2 / 50 + 12 ** 2 / 40)                 # P1.9
p_pool = (120 + 150) / (1000 + 1000)                             # P1.10

print(f"SE {se}   z {z}   p {p:.4f}   critical ±{crit:.3f}   t {t:.3f} (df {df})")
print(f"z for the proportion {z_prop:.2f}   SE of the difference {se_diff:.3f}   pooled p̄ {p_pool}")
for k, v in (("P1.1", [se]), ("P1.4", [z]), ("P1.5", [p]), ("P1.6", [crit]), ("P1.7", [t, df]), ("P1.8", [z_prop]),
             ("P1.9", [se_diff]), ("P1.10", [p_pool])):
    assert agrees(k, v), k

### Slide the sample mean: watch the p-value

The course's battery cells: the maker claims at least 500 charge cycles, σ = 40, and a sample of 64 cells averages
488. Under H₀ the standard error is 40/√64 = 5, so z = −12/5 = −2.4, and the p-value is the gold area to the left:
about 0.0082. Slide x̄ towards 500 and watch the area grow; change n and watch the standard error.

In [ ]:
def pvalue_picture(xbar=488.0, n=64):
    se = 40 / math.sqrt(n)
    z = (xbar - 500) / se
    p = norm.cdf(z)
    zs = np.linspace(-4, 4, 400)
    fig, ax = plt.subplots(figsize=(11.5, 4))
    fig.subplots_adjust(left=0.05, right=0.98, top=0.8, bottom=0.18)
    ax.plot(zs, norm.pdf(zs), color=CHALK)
    lo = np.linspace(-4, max(-4, min(z, 4)), 200)
    ax.fill_between(lo, norm.pdf(lo), color=GOLD, alpha=0.8, lw=0)
    ax.axvline(norm.ppf(0.05), color=VIOLET, ls="--", lw=1.6)
    ax.text(norm.ppf(0.05), 0.42, " critical −1.645", color=VIOLET, fontsize=12)
    ax.set_xlim(-4, 4); ax.set_ylim(0, 0.45); ax.set_yticks([]); ax.set_xlabel("z, in standard errors from 500")
    verdict = "reject H₀" if p <= 0.05 else "do not reject H₀"
    fig.suptitle(f"x̄ = {xbar:g}, n = {n}: SE = {se:.2f}, z = {z:.2f}, p = {p:.4f} → {verdict}", x=0.05, ha="left",
                 fontsize=15, color=GOLD)
    footnote(fig); plt.show()

assert 40 / math.sqrt(64) == 5 and (488 - 500) / 5 == -2.4 and round(norm.cdf(-2.4), 4) == 0.0082   # the video's numbers

In [ ]:
interact(pvalue_picture, xbar=FloatSlider(value=488, min=480, max=505, step=0.5, description="x̄ (cycles)"),
         n=IntSlider(value=64, min=16, max=256, step=4, description="n"));

## Tier 2 · Core

The module's standard tests, several steps each: a z test against a claim, a two-tailed test with its interval, t
tests from a summary and from raw data, a proportion against a claim, β and power, a sample size, Welch's test from
two summaries, a matched-samples test, and two proportions.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**The t distribution.** `stats.t.sf(t, df)` is the right tail (T.DIST.RT), `stats.t.cdf` the left tail, and
`stats.t.ppf(0.95, df)` a critical value (T.INV). Unlike a spreadsheet, these take a fractional df as it is.

**Tests from raw data.** `stats.ttest_1samp(x, mu0)` tests one mean; `stats.ttest_rel(after, before)` a matched pair
of columns; `stats.ttest_ind(a, b, equal_var=False)` Welch's test of two independent samples. Each returns the
statistic and a two-tailed p-value; pass `alternative="greater"` or `"less"` for one tail. From summary numbers,
`stats.ttest_ind_from_stats(m1, s1, n1, m2, s2, n2, equal_var=False)`.

In [ ]:
print("right tail of t = 2 with 24 df:", stats.t.sf(2, 24))                            # the video's 0.0285
assert round(stats.t.sf(2, 24), 4) == 0.0285 and round(stats.t.ppf(0.95, 24), 3) == 1.711

strength = [512, 526, 519, 508, 515, 511, 523, 506]                                    # the video's tensile tests
print("tensile strength, H₀ μ = 520:", stats.ttest_1samp(strength, 520))
r = stats.ttest_1samp(strength, 520)
assert round(r.statistic, 2) == -1.98 and round(r.pvalue, 3) == 0.088

sensor = [100.4, 99.8, 100.9, 100.6, 100.2, 101.1, 99.9, 100.7]                       # P2.4
r = stats.ttest_1samp(sensor, 100.0)
print("P2.4 sensor:", round(r.statistic, 3), round(r.pvalue, 4))
assert agrees("P2.4", [r.statistic, r.pvalue])

t = (43.1 - 40) / (6.2 / math.sqrt(20))                                                # P2.3, from the summary
assert agrees("P2.3", [t, stats.t.sf(t, 19)])

r = stats.ttest_ind_from_stats(6.8, 1.2, 15, 6.1, 0.9, 18, equal_var=False)           # P2.8: Welch from summaries
v1, v2 = 1.2 ** 2 / 15, 0.9 ** 2 / 18
df_w = (v1 + v2) ** 2 / (v1 ** 2 / 14 + v2 ** 2 / 17)                                  # Welch's degrees of freedom
print("P2.8 Welch:", round(r.statistic, 3), "df", round(df_w, 1), "p", round(r.pvalue, 4))
assert agrees("P2.8", [r.statistic, df_w, r.pvalue])

before = [42, 38, 51, 45, 36, 48, 40, 55, 44, 39]                                      # P2.9: matched
after = [45, 41, 50, 49, 40, 52, 41, 58, 47, 41]
r = stats.ttest_rel(after, before, alternative="greater")
d = np.array(after) - np.array(before)
print("P2.9 matched: d̄", d.mean(), " t", round(r.statistic, 2), " p", round(r.pvalue, 4))
assert agrees("P2.9", [d.mean(), r.statistic, r.pvalue])

crit = 120 + norm.ppf(0.95) * 15 / math.sqrt(25)                                       # P2.6: β and power
beta = norm.cdf((crit - 127) / 3)
assert agrees("P2.6", [beta, 1 - beta])
n = math.ceil((norm.ppf(0.95) + norm.ppf(0.90)) ** 2 * 8 ** 2 / 3 ** 2)                # P2.7
assert agrees("P2.7", [n])

In [ ]:
# Welch's test on the checkout pages' order values, in three lines (the video's 0.036)
old = [44, 61, 38, 55, 47, 66, 52, 41, 58, 49, 63, 50]
new = [57, 72, 49, 66, 51, 70, 59, 63, 52, 77, 46, 68, 54, 56]
stats.ttest_ind(new, old, equal_var=False)

In [ ]:
r = stats.ttest_ind(new, old, equal_var=False)
assert round(r.pvalue, 3) == 0.036 and round(r.statistic, 2) == 2.22

### The power curve: n is the lever

The battery test again (H₀: μ ≥ 500, σ = 40, α = 0.05, lower tail). For each true mean, the power is the chance the
test rejects H₀. With n = 64 the power at 490 is about 0.64; the course's sample-size formula says n = 138 lifts it to
about 0.90. Slide n and watch the curve steepen: the critical value moves towards 500 as the standard error shrinks.

In [ ]:
def power_picture(n=64):
    se = 40 / math.sqrt(n)
    crit = 500 - norm.ppf(0.95) * se
    mus = np.linspace(470, 505, 300)
    power = norm.cdf((crit - mus) / se)
    p490 = norm.cdf((crit - 490) / se)
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.82, bottom=0.16)
    ax.plot(mus, power, color=SKY)
    ax.axhline(0.05, color=VIOLET, ls=":", lw=1.4); ax.text(470.5, 0.08, "α = 0.05 at μ = 500", color=VIOLET, fontsize=12)
    ax.scatter([490], [p490], s=110, color=GOLD, zorder=4)
    ax.set_xlim(470, 505); ax.set_ylim(0, 1.05); ax.set_xlabel("the true mean (cycles)"); ax.set_ylabel("power")
    fig.suptitle(f"n = {n}: SE = {se:.2f}, reject if x̄ < {crit:.1f}; power at 490 = {p490:.2f}", x=0.07, ha="left",
                 fontsize=15, color=GOLD)
    footnote(fig); plt.show()

_pw = lambda n: norm.cdf((500 - norm.ppf(0.95) * 40 / math.sqrt(n) - 490) / (40 / math.sqrt(n)))
assert round(_pw(64), 2) == 0.64 and round(_pw(138), 2) == 0.90                         # the video's numbers
assert math.ceil((norm.ppf(0.95) + norm.ppf(0.90)) ** 2 * 40 ** 2 / 10 ** 2) == 138

In [ ]:
interact(power_picture, n=IntSlider(value=64, min=16, max=300, step=2, description="n"));

## Tier 3 · Stretch

Combine ideas, choose the method, spot the trap, work backwards: a tail switched after seeing the data, a test read
from a confidence interval, the smallest α that rejects, power with a bigger sample, significant but tiny, pooled
against Welch with unequal spreads, ignoring the pairing, and the visitors an A/B test needs.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

`stats.ttest_ind(a, b)` with no other argument is the pooled test, which assumes one variance (a spreadsheet's
T.TEST type 2); `equal_var=False` is Welch's (type 3). Run both on the suppliers of P3.6 and see how far apart they
land. `stats.ttest_rel` is the matched test (type 1): on the same columns it can find what an independent-samples
test misses, because it removes the spread between people, servers or stores.

In [ ]:
a = [82, 104, 95, 58, 116, 74, 93, 101, 67, 80]                                       # P3.6
b = [78, 74, 81, 69, 76, 72, 80, 75, 77, 73, 79, 70, 82, 74, 76, 71, 78, 75, 73, 77]
pooled, welch = stats.ttest_ind(a, b), stats.ttest_ind(a, b, equal_var=False)
va, vb = np.var(a, ddof=1) / len(a), np.var(b, ddof=1) / len(b)
df_w = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
print(f"pooled: t = {pooled.statistic:.2f}, p = {pooled.pvalue:.4f}   Welch: t = {welch.statistic:.2f}, "
      f"df = {df_w:.1f}, p = {welch.pvalue:.4f}")
assert agrees("P3.6", [pooled.pvalue, welch.pvalue, df_w])

print("P3.7, ignoring the pairing:", round(stats.ttest_ind(after, before, equal_var=False).pvalue, 4),
      "  matched:", round(stats.ttest_rel(after, before).pvalue, 4))
assert agrees("P3.7", [stats.ttest_ind(after, before, equal_var=False).pvalue])

current = [182, 205, 167, 221, 194, 176, 210, 188]                                    # the video's eight servers
release = [171, 199, 160, 207, 190, 163, 204, 179]
print("servers, related-samples t test:", stats.ttest_rel(current, release))
print("servers, as if independent     :", stats.ttest_ind(current, release, equal_var=False))
assert round(stats.ttest_rel(current, release).pvalue, 4) == 0.0002
assert round(stats.ttest_ind(current, release, equal_var=False).pvalue, 2) == 0.36

z_a, z_b = norm.ppf(1 - 0.05 / 2), norm.ppf(0.80)                                      # P3.8: visitors per version
n = math.ceil((z_a + z_b) ** 2 * (0.05 * 0.95 + 0.06 * 0.94) / 0.01 ** 2)
assert agrees("P3.8", [n])
se = 25 * math.sqrt(2 / 400000)                                                        # P3.5: significant but tiny
print(f"P3.5: z = {0.15 / se:.2f}, interval ${0.15 - z_a * se:.2f} to ${0.15 + z_a * se:.2f}: below the $0.50 that matters")
assert agrees("P3.5", [2 * norm.sf(0.15 / se), 0.15 - z_a * se, 0.15 + z_a * se])

### The same lift, more visitors

The course's opening question: a new checkout page sells to 4.6% of visitors, the old one to 4.0%. With 6,000
visitors a page, p ≈ 0.105: it could be luck. With 20,000, p ≈ 0.003. The lift never changed; the standard error
did. Slide the visitors and watch the p-value cross 0.05.

In [ ]:
def ab_picture(visitors=6000):
    def p_of(v):
        pb = (0.04 + 0.046) / 2
        return 2 * norm.sf(0.006 / np.sqrt(pb * (1 - pb) * 2 / v))
    vs = np.linspace(1000, 40000, 400)
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.82, bottom=0.16)
    ax.plot(vs, p_of(vs), color=SKY)
    ax.axhline(0.05, color=CORAL, ls=":", lw=1.6); ax.text(30500, 0.07, "p = 0.05", color=CORAL, fontsize=12)
    ax.scatter([visitors], [p_of(visitors)], s=120, color=GOLD, zorder=4)
    ax.set_xlim(1000, 40000); ax.set_ylim(0, 0.6); ax.set_xlabel("visitors a page"); ax.set_ylabel("two-tailed p-value")
    fig.suptitle(f"4.0% vs 4.6% with {visitors:,} visitors a page: p = {p_of(visitors):.3f}", x=0.08, ha="left",
                 fontsize=15, color=GOLD)
    footnote(fig); plt.show()

def two_prop_z(x1, n1, x2, n2):
    pb = (x1 + x2) / (n1 + n2)
    return (x2 / n2 - x1 / n1) / math.sqrt(pb * (1 - pb) * (1 / n1 + 1 / n2))

assert round(2 * norm.sf(two_prop_z(240, 6000, 276, 6000)), 3) == 0.105                  # the video's numbers
assert round(2 * norm.sf(two_prop_z(800, 20000, 920, 20000)), 3) == 0.003

In [ ]:
interact(ab_picture, visitors=IntSlider(value=6000, min=1000, max=40000, step=500, description="visitors"));

## Tier 4 · The case: Wrenmoor Energy's in-home display

Wrenmoor Energy is an electricity utility (a fictional company; our own data). It gave an in-home display (live
usage and cost, plus a weekly report) to 200 of 400 households chosen at random; the other 200 are the control
group. The question: does the display cut usage enough to pay for itself? Before the trial, the manager set the bar:
a saving of 15 kWh a month pays the display back within two years, at $0.16 per kWh, $45 per display and $0.50 a
month for the report. Here are the 400 households, one row each:

| column | what it is |
|---|---|
| `household` | household id, H001–H400 |
| `group` | display (given the in-home display) or control, at random |
| `home` | house or apartment |
| `before_kwh` | kWh a month, the same three winter months a year earlier |
| `after_kwh` | kWh a month, the three winter months of the trial |

The next cell builds the table from a seeded random generator, so everyone gets exactly the same 400 households
(the same as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
CASE_SEED = 2043
PRICE_KWH, DISPLAY_COST, REPORT_COST, THRESHOLD_KWH = 0.16, 45.0, 0.50, 15

def build_case(seed=CASE_SEED):
    """400 households, alternately display / control, shuffled; usage a year before and during the trial."""
    r = random.Random(seed)
    rows = []
    for i in range(400):
        g = "display" if i % 2 == 0 else "control"
        home = "house" if r.random() < 0.55 else "apartment"
        base = 1050 if home == "house" else 640
        before = int(round(max(250, r.gauss(base, 0.28 * base))))
        after = int(round(before - 0.055 * before - (0.035 * before if g == "display" else 0.0) + r.gauss(0, 55)))
        rows.append((g, home, before, after))
    r.shuffle(rows)
    return pd.DataFrame({"household": [f"H{k + 1:03d}" for k in range(400)], "group": [x[0] for x in rows],
                         "home": [x[1] for x in rows], "before_kwh": [x[2] for x in rows],
                         "after_kwh": [x[3] for x in rows]})

df = build_case()

# The case's numbers as the course computed them, to check this table against.
# @STATS
CASE_STATS = {'n_display': 200,
 'n_control': 200,
 'n_house': 227,
 'n_apartment': 173,
 'before_display': 872.4,
 'before_control': 865.355,
 'after_display': 793.62,
 'after_control': 816.925,
 'change_display': -78.78,
 'change_control': -48.43,
 'saving': 30.35,
 'change_se': 6.185810761239736,
 'change_t': -4.906389990164745,
 'change_df': 397.21319852375655,
 'ci': (18.188979344747743, 42.51102065525226),
 'th_t': 2.4814855469202253,
 'payback_mid': 10.330578512396693,
 'payback_low': 18.67036548334498}
# @END STATS
S = CASE_STATS
assert len(df) == 400 and (df["group"] == "display").sum() == S["n_display"] == 200
assert (df["home"] == "house").sum() == S["n_house"]
g = df.groupby("group")
assert abs(g["before_kwh"].mean()["display"] - S["before_display"]) < 1e-9
assert abs(g["after_kwh"].mean()["control"] - S["after_control"]) < 1e-9
print(f"{len(df)} households, {df.shape[1]} columns: the same data as the course's case.")
df.head(8)

### How you'd do this in Python

With the households in a DataFrame, each step of the case is a line or two: a `change` column (after − before),
`groupby("group")` to compare the display and control groups, and `stats.ttest_ind(…, equal_var=False)` for Welch's
test. The cells below reproduce the video's case: the two tempting analyses that mislead, then the right one, which
compares each group's change.

In [ ]:
df["change"] = df["after_kwh"] - df["before_kwh"]
D, C = df[df["group"] == "display"], df[df["group"] == "control"]
print(df.groupby("group")[["before_kwh", "after_kwh", "change"]].agg(["mean", "std"]).round(2).to_string(), "\n")

after_only = stats.ttest_ind(D["after_kwh"], C["after_kwh"], equal_var=False)            # tempting analysis 1
print(f"1 · this winter only, display vs control: p = {after_only.pvalue:.2f}  (the spread between homes hides it)")
before_after = stats.ttest_1samp(D["change"], 0)                                         # tempting analysis 2
print(f"2 · display homes, before vs after: t = {before_after.statistic:.1f}  (the mild winter credited to the display)")
right_one = stats.ttest_ind(D["change"], C["change"], equal_var=False, alternative="less")   # the right one
saving = C["change"].mean() - D["change"].mean()
print(f"3 · compare the changes: extra saving {saving:.2f} kWh a month, t = {right_one.statistic:.2f}, "
      f"p = {right_one.pvalue:.1e}")
assert round(after_only.pvalue, 2) == 0.44 and round(before_after.statistic, 1) == -17.6
assert round(saving, 2) == 30.35 and round(right_one.statistic, 2) == -4.91

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.6))
fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.15)
rng = np.random.default_rng(1)
for k, (name, part) in enumerate((("display", D), ("control", C))):
    ax.scatter(part["change"], k + rng.uniform(-0.18, 0.18, len(part)), s=16, color=PAIR[name], alpha=0.7)
    ax.plot([part["change"].mean()] * 2, [k - 0.3, k + 0.3], color=PAIR[name], lw=3)
    ax.text(part["change"].mean(), k + 0.36, f"mean {part['change'].mean():.2f}", color=PAIR[name], ha="center")
ax.axvline(0, color=DIM, lw=1)
ax.set_yticks([0, 1], ["display", "control"]); ax.set_ylim(-0.6, 1.7); ax.set_xlabel("change in use, kWh a month")
fig.suptitle(f"Both groups used less (a milder winter); the display group {saving:.2f} kWh a month more",
             x=0.1, ha="left", fontsize=15, color=GOLD)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: each type of home's saving, the share of homes that cut their
use by 5% or more, and the payback at the cautious end of the interval, ending in your recommendation. Use the
DataFrame `df` in a cell of your own (press **+** to add one), then type your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `df` with pandas and scipy and checks it
against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
def welch(x, y):
    """(SE, Welch's degrees of freedom) for the difference of two means."""
    vx, vy = x.var(ddof=1) / len(x), y.var(ddof=1) / len(y)
    return math.sqrt(vx + vy), (vx + vy) ** 2 / (vx ** 2 / (len(x) - 1) + vy ** 2 / (len(y) - 1))

mine = {}
mine["P4.1"] = [D["change"].mean(), C["change"].mean(), saving]
mine["P4.2"] = [stats.ttest_ind(D["after_kwh"], C["after_kwh"], equal_var=False).pvalue]
se, dfw = welch(D["change"], C["change"])
t = (D["change"].mean() - C["change"].mean()) / se
mine["P4.3"] = [t, dfw, stats.t.cdf(t, dfw)]
q = stats.t.ppf(0.975, dfw)
lo, hi = saving - q * se, saving + q * se
mine["P4.4"] = [lo, hi]
t_bar = (saving - THRESHOLD_KWH) / se
mine["P4.5"] = [t_bar, stats.t.sf(t_bar, dfw)]
seg = {}
for home in ("house", "apartment"):
    d_, c_ = D[D["home"] == home]["change"], C[C["home"] == home]["change"]
    s_, df_ = welch(d_, c_)
    sv = c_.mean() - d_.mean(); qq = stats.t.ppf(0.975, df_)
    usual = df[df["home"] == home]["before_kwh"].mean()
    seg[home] = sv
    print(f"{home:9s}: extra saving {sv:.1f} kWh a month ({sv - qq * s_:.1f} to {sv + qq * s_:.1f}), "
          f"{100 * sv / usual:.1f}% of usual use")
mine["P4.6"] = [seg["house"], seg["apartment"]]
cut_d, cut_c = (D["after_kwh"] <= 0.95 * D["before_kwh"]).mean(), (C["after_kwh"] <= 0.95 * C["before_kwh"]).mean()
z = two_prop_z(cut_c * 200, 200, cut_d * 200, 200)
mine["P4.7"] = [cut_d, cut_c, z, norm.sf(z)]
mine["P4.8"] = [DISPLAY_COST / (lo * PRICE_KWH - REPORT_COST)]

for k, got in mine.items():
    assert agrees(k, [float(v) for v in got]), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(float(v), 6))}" for p, v in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")
print(f"Payback: about {DISPLAY_COST / (saving * PRICE_KWH - REPORT_COST):.1f} months at the middle estimate, "
      f"{mine['P4.8'][0]:.1f} at the interval's low end: inside the 24-month bar either way.")

## Sources
Every exercise, data set and company here is our own (Wrenmoor Energy is fictional). The chapter order follows a
standard first course in business statistics (hypothesis tests for one population, then for two); the definitions
agree with OpenStax, Introductory Statistics 2e and Introductory Business Statistics 2e, chapters 9 and 10 (retrieved
2026-11-07).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs04_hypothesis_testing/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs04_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"]) == (_e["q"], _e["working"], _e["function"]), _e["id"]
        assert all((a == b) if isinstance(b, str) else abs(a - float(b)) < 1e-9
                   for a, b in zip(_q["answer"], _flat(_e["answer"]))) and len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    assert _P.CASE["before_kwh"] == df["before_kwh"].tolist() and _P.CASE["after_kwh"] == df["after_kwh"].tolist()
    assert _P.CASE["group"] == df["group"].tolist() and _P.CASE["home"] == df["home"].tolist()
    print("matches the course's practice set")